In [163]:
import struct

def read_variable_length(data, offset):
    # reads a variable acording to the midi the bottom 7 bits are used for value
    # if the 8th bit is 0 the value is done if the 8th bit is 1 grabs one more byte
    value = 0
    while True:
        byte = data[offset]
        offset += 1
        value = (value << 7) | (byte & 0x7F)
        if not (byte & 0x80):
            break
    return value, offset
    
def readMidiFile(path):
    src = open(path, "rb")
    current_time = 0
    while True:
        chunkID = src.read(4)
        chunklen = int.from_bytes(src.read(4))
        chunk = src.read(chunklen)
        header_dict = {}
        delta_times = False
        if chunkID == b'MThd':
            header_dict['format'], header_dict['ntracks'], header_dict['tickdiv']  = struct.unpack('>HHH', chunk[0:6])
            if header_dict['format'] != 1:
                    print('Only format 1 MIDI files are suported')
            print('Header: ', header_dict)
        elif chunkID == b'MTrk':
            offset = 0
            print("track:")
            #print(chunk)
            #b'\x00\xff\x03\x00\x00\xffX\x04\x04\x02\x18\x08\x00\xffQ\x03\x08,\xa2\x00\xff/\x00'

            while offset < chunklen:
                # Read delta time
                delta, offset = read_variable_length(chunk, offset)
                current_time += delta
                event_status = chunk[offset]
                offset += 1
                if event_status == 0xFF: #Meta event
                    meta_type = chunk[offset]
                    offset += 1
                    meta_length = chunk[offset]
                    offset += 1
                    meta_data = chunk[offset:offset+meta_length]
                    offset += meta_length
                    if meta_type == 0x2F and meta_data == b'':
                        print(current_time,"- Metadata > End of track")
                    elif meta_type == 0x03:
                        print(current_time,"- Metadata > Track/Sequence name > ",meta_data)
                    elif meta_type == 0x51:
                        tempo = struct.unpack('>i', b'\x00'+meta_data)[0]
                        print(current_time,"- Metadata > Tempo > ",tempo)
                    elif meta_type == 0x58:
                        num, denom, clocks_per_clck, notes_per_beat = struct.unpack('>bbbb', meta_data)
                        print(current_time,"- Metadata > Time signature >",num, denom, clocks_per_clck, notes_per_beat)
                    else:
                        print(current_time,'- type - ',meta_type,' - Meta Data > ',meta_data)
                elif event_status//16 == 9 or event_status//16 == 8:
                    note = chunk[offset]
                    offset += 1
                    vel = chunk[offset]
                    offset += 1
                    if event_status//16 == 9:
                        print(current_time,' - Note_On > n:',note,' v:',vel)
                    else:
                        print(current_time,' - Note_Off > n:',note,' v:',vel)
                elif event_status//16 == 11:
                    controller = chunk[offset]
                    offset += 1
                    value = chunk[offset]
                    offset += 1
                    print(current_time,' - controller > ctrl: ',controller,'val: ',value)
                elif event_status//16 == 12:
                    pgrm_n = chunk[offset]
                    offset += 1
                    print(current_time,' - program > pgrm_n: ',pgrm_n)
                elif event_status//16 == 14:
                    lsb = chunk[offset]
                    offset += 1
                    msb = chunk[offset]
                    offset += 1
                    print(current_time,' - pitchBend > lsb: ',lsb,' msb: ',msb)
                else:
                    print(current_time,' - ',hex(event_status))
        else:
            break

In [164]:
mid = readMidiFile('../ChipTune Melodies/Kid Dracula - Go-Go at the Great Castle.mid')

Header:  {'format': 1, 'ntracks': 2, 'tickdiv': 480}
track:
0 - Metadata > Track/Sequence name >  b''
0 - Metadata > Time signature > 4 2 24 8
0 - Metadata > Tempo >  535714
0 - Metadata > End of track
track:
0  - controller > ctrl:  121 val:  0
0 - Metadata > Track/Sequence name >  b'main track'
0  - controller > ctrl:  10 val:  64
0  - controller > ctrl:  7 val:  100
0  - controller > ctrl:  11 val:  127
0  - controller > ctrl:  101 val:  0
0  - controller > ctrl:  100 val:  2
0  - controller > ctrl:  6 val:  64
0  - controller > ctrl:  101 val:  0
0  - controller > ctrl:  100 val:  1
0  - controller > ctrl:  6 val:  64
0  - controller > ctrl:  38 val:  0
0  - controller > ctrl:  101 val:  0
0  - controller > ctrl:  100 val:  0
0  - controller > ctrl:  6 val:  12
0  - pitchBend > lsb:  0  msb:  64
0  - controller > ctrl:  1 val:  0
0  - program > pgrm_n:  80
720  - Note_On > n: 85  v: 100
960  - Note_Off > n: 85  v: 0
960  - Note_On > n: 83  v: 100
1200  - Note_Off > n: 83  v: 0
1200

In [68]:
from mido import MidiFile

# Load the MIDI file
mid = MidiFile('../ChipTune Melodies/Kid Dracula - Go-Go at the Great Castle.mid')

# Print file metadata
print(f"Type: {mid.type}, Ticks per beat: {mid.ticks_per_beat}")

# Iterate through tracks and messages
for track in mid.tracks:
    for msg in track:
        print(msg)

Type: 1, Ticks per beat: 480
MetaMessage('track_name', name='', time=0)
MetaMessage('time_signature', numerator=4, denominator=4, clocks_per_click=24, notated_32nd_notes_per_beat=8, time=0)
MetaMessage('set_tempo', tempo=535714, time=0)
MetaMessage('end_of_track', time=0)
control_change channel=0 control=121 value=0 time=0
MetaMessage('track_name', name='main track', time=0)
control_change channel=0 control=10 value=64 time=0
control_change channel=0 control=7 value=100 time=0
control_change channel=0 control=11 value=127 time=0
control_change channel=0 control=101 value=0 time=0
control_change channel=0 control=100 value=2 time=0
control_change channel=0 control=6 value=64 time=0
control_change channel=0 control=101 value=0 time=0
control_change channel=0 control=100 value=1 time=0
control_change channel=0 control=6 value=64 time=0
control_change channel=0 control=38 value=0 time=0
control_change channel=0 control=101 value=0 time=0
control_change channel=0 control=100 value=0 time=0


In [116]:
b'0'

b'0'

In [117]:
bytearray(0x00)

bytearray(b'')

In [134]:
0x91//16

9